In [12]:
# for tag in soup.find_all(string=lambda x: x and 'Domicílio Eletrônico' in x):
#     print(tag)
#     print(tag.parent)
import re
import json
from pathlib import Path
import pandas as pd
from pprint import pprint

In [13]:
def extrair_meio(texto):

    texto = texto.lower()

    if any(x in texto for x in ('advgs', '(P/ Advgs.', 'advgs.', 'advogado')):
        return 'advogado'

    if 'ofício' in texto or 'oficio' in texto:
        return 'oficio'

    if 'mandado' in texto:
        return 'mandado'
    # if 'prazo' in texto:
    #     return 'mandado'

    # if 'ofício' in texto:
    #     return 'oficio'

    if ('aviso de recebimento' in texto or 'juntada de ar' in texto):
        return 'ar'
    # if '(para ' in texto:
    #     return 'pessoal'
    if re.search(r'\bpara\b', texto):
        return 'pessoal'

    return None

def situacao_comunicacao(texto):

    texto = str(texto).lower()

    if 'lido(a)' in texto:
        return 'lida'

    if 'expedido(a)' in texto:
        return 'expedida'

    if 'devolução sem leitura' in texto:
        return 'devolvida_sem_leitura'

    if 'juntada de ar' in texto:
        return 'ar_juntado'

    if 'mandado devolvido' in texto:
        return 'mandado_devolvido'

    if 'mandado assinado' in texto:
        return 'mandado_assinado'

    if 'mandado à disposição' in texto:
        return 'mandado_disponivel'

    if 'solicitada a expedição de mandado' in texto:
        return 'mandado_solicitado'

    return None

def tipo_comunicacao(texto):
    texto = texto.lower()

    if 'citação' in texto:
        return 'citacao'

    if 'intimação' in texto:
        return 'intimacao'

    if 'certidão' in texto:
        return 'certidao'
    if 'juntada de ar' in texto:
        return 'certidao'

    return 'outro'

def expedidas(df):
    expedidas = df[df['situacao_comunicacao'] == 'expedida'].copy()
    expedidas['meio_real'] = expedidas['ato'].apply(extrair_meio)
    return expedidas

def referenciados(df):
    return df[df['observacao'].str.contains('Referente ao evento', case=False, na=False)].copy()

def lidas (df):

    lidas = (df[df['situacao_comunicacao'] == 'lida']).copy()
    # mask_ar = df[df['ato'].str.contains(r'juntada de ar|aviso de recebimento', case=False, na=False)].copy()
    # len(lidas)
    # print(mask_lida.sum())
    # print(mask_ar.sum())
    # return df[mask_lida|mask_ar]
    return lidas
    # return pd.concat([mask_lida, mask_ar]).drop_duplicates()

def chave(df_expedidas, df_lidas):
    df_expedidas['chave'] = (
        df_expedidas['destinatario']
        .str.lower()
        .str.strip()
        + '|'
        + df_expedidas['data_texto']
    )

    df_lidas['chave'] = (
       df_lidas['destinatario']
        .str.lower()
        .str.strip()
        + '|'
        + df_lidas['data_referencia_str']
    )

# print(len(expedidas), len(lidas))

def relacoes(df_expedidas, df_lidas):
    relacoes = df_lidas.merge(
        df_expedidas,
        on=['chave', 'destinatario', 'tipo'],
        suffixes=(
            '_lido',
            '_expedido'
        )
    )
    return relacoes

# def origem(df_relacoes):
#     relacoes['ato_origem'] = relacoes['ato_lido'].str.extract(
#         r'Referente ao evento\s+(.*?)\('
#     )

#     relacoes['data_origem'] = relacoes['ato_lido'].str.extract(
#         r'\((\d{2}/\d{2}/\d{2})\)'
#     )
def origem(df_relacoes):

    df_relacoes['ato_origem'] = (
        df_relacoes['ato_lido']
        .str.extract(
            r'Referente ao evento\s+(.*?)\(',
            expand=False
        )
    )

    df_relacoes['data_origem'] = (
        df_relacoes['ato_lido']
        .str.extract(
            r'\((\d{2}/\d{2}/\d{2})\)',
            expand=False
        )
    )

    return df_relacoes
# def canais_disponiveis(row):

#     canais = []

#     if row['tem_advogado']:
#         canais.append('advogado')

#     if row['domicilio_cnj']:
#         canais.append('domicilio_cnj')

#     if pd.notna(row['email']):
#         canais.append('email')

#     return ', '.join(canais)
def canais_disponiveis(row):

    canais = []

    if row['tem_advogado']:
        canais.append('advogado')

    if row['domicilio_cnj']:
        canais.append('domicilio_cnj')

    if row['recebe_intimacao_email']:
        canais.append('email_autorizado')

    elif pd.notna(row['email']):
        canais.append('email_cadastrado')

    if pd.notna(row['tel']):
        canais.append('telefone')

    return ', '.join(canais)


def lista_canais_disponiveis(row):

    canais = []

    if row['tem_advogado']:
        canais.append('advogado')

    if row['domicilio_cnj']:
        canais.append('domicilio_cnj')

    if row['recebe_intimacao_email']:
        canais.append('email_autorizado')

    elif pd.notna(row['email']):
        canais.append('email_cadastrado')

    if pd.notna(row['tel']):
        canais.append('telefone')

    return canais


def status_comunicacao(row):

    if row['tem_advogado']:
        return 'alta_advogado'

    if row['domicilio_cnj']:
        return 'alta_domicilio_cnj'

    if row['recebe_intimacao_email']:
        return 'alta_email_autorizado'

    if pd.notna(row['email']):
        return 'media_email_cadastrado'

    return 'baixa_sem_canal_eletronico'

# def status_comunicacao(row):

#     canais = row['canais_disponiveis']

#     if not canais:
#         return 'baixa'

#     if all(x in canais for x in ['advogado', 'domicilio_cnj', 'email']):
#         return 'muito_alta'

#     if 'advogado' in canais and 'domicilio_cnj' in canais:
#         return 'alta_adv_djen'

#     if 'domicilio_cnj' in canais and 'email' in canais:
#         return 'alta_djen_email'

#     if 'advogado' in canais:
#         return 'media_advogado'

#     if 'domicilio_cnj' in canais:
#         return 'media_djen'

#     if 'email' in canais:
#         return 'media_email'

#     return 'baixa'

# def status_processo(processo_ok, autores_ok, acusados_ok):
#     if processo_ok:
#         return 'automatizar'

#     if autores_ok and not acusados_ok:
#         return 'pendencia_promovido'

#     if acusados_ok and not autores_ok:
#         return 'pendencia_promovente'

#     return 'pendencia_geral'
def status_processo(autores_ok, acusados_ok):

    if autores_ok and acusados_ok:
        return 'automatizar'

    if not autores_ok and acusados_ok:
        return 'autor_pendente'

    if autores_ok and not acusados_ok:
        return 'reu_pendente'

    return 'ambos_pendentes'


def normalizar_nome(x):
    if pd.isna(x):
        return None

    x = x.lower().strip()
    x = re.sub(r'\(rev\.\s*arg\.?\)', '', x, flags=re.I)
    x = re.sub(r'\s+', ' ', x)
    return x

def marcar_revelia(nome):
    if pd.isna(nome):
        return False
    return bool(re.search(r'\(rev\.\s*arg\.?\)', nome, re.I))

def status_processo(df_partes):

    autores_ok = df_partes.loc[
        df_partes['papel'] == 'PROMOVENTE',
        'habilitada_receber'
    ].all()

    acusados_ok = df_partes.loc[
        df_partes['papel'] == 'PROMOVIDO',
        'habilitada_receber'
    ].all()

    if autores_ok and acusados_ok:
        return True, 'automatizar'

    if autores_ok and not acusados_ok:
        return False, 'autor_pendente'

    if not autores_ok and acusados_ok:
        return False, 'reu_pendente'

    return False, 'ambos_pendentes'

padrao_revelia = re.compile(r'\(rev\.\s*arg\.?\)', re.I)

def tratar_parte(parte):
    nome_original = parte['nome']
    nome_norm_original = parte.get('nome_normalizado', nome_original)

    revelia = bool(padrao_revelia.search(nome_original))

    return {
        **parte,
        "revelia": revelia,
        "nome_limpo": padrao_revelia.sub("", nome_original).strip(),
        "nome_normalizado_limpo": padrao_revelia.sub("", nome_norm_original).strip()
    }

In [14]:
def fisicos_expedidos(df):

    mask = df['ato'].str.contains(
        r'mandado assinado|ofício assinado|carta assinado',
        case=False,
        na=False
    )

    return df[mask].copy()

def fisicos_retorno(df):

    mask = df['ato'].str.contains(
        r'mandado devolvido|ofício juntado|juntada de ar',
        case=False,
        na=False
    )

    return df[mask].copy()

In [15]:
pasta = Path("C:\\Users\\igusilva\\OneDrive - Tribunal de Justiça do Estado da Bahia\\Área de Trabalho\\assets\\apps\\vitoria_secret\\udemy\\data_science\\nlp_spacy\\processos_movs_json")

In [16]:
arquivos = list(pasta.glob('*.json'))

print(len(arquivos))

43


In [21]:
partes_processos = {}
for arquivo in pasta.glob("*.json"):
    processo = arquivo.stem

    # print("Lendo:", arquivo)

    try:

        with open(arquivo, "r", encoding="utf-8") as f:
            dados = json.load(f)
            partes = [tratar_parte(p) for p in dados['partes']]
            df_atos = pd.DataFrame(dados['movimentacoes'])
            df_partes = pd.DataFrame(partes)

            print(df_atos[
                df_atos['ato'].str.contains(
                    r'mandado|ofício|ar',
                    case=False,
                    na=False
                )
            ][['evento', 'ato', 'data_texto']])
           
    except Exception as e:

        print("ERRO:", e)

   evento                                                ato data_texto
0      82  Intimação expedido(a) (P/ Advgs. de PAYOUT PAG...   02/06/26
1      81  Intimação expedido(a) (P/ Advgs. de DANILO AUG...   02/06/26
2      72  Intimação expedido(a) (P/ Advgs. de DANILO AUG...   01/06/26
3      62  Intimação lido(a) (Para TV VINHOS LTDA) em 11/...   17/05/26
4      61  Intimação lido(a) (Para PAYOUT PAGAMENTOS INTE...   09/05/26
5      60  Intimação expedido(a) Para TV VINHOS LTDA *Ref...   28/04/26
6      58  Intimação expedido(a) (Para PAYOUT PAGAMENTOS ...   28/04/26
7      49  Intimação expedido(a) (P/ Advgs. de DANILO AUG...   17/04/26
8      37  Intimação expedido(a) (P/ Advgs. de DANILO AUG...   20/02/26
9      33  Audiência de Conciliação Realizada (Telepresen...   19/02/26
11     28  Intimação lido(a) (Para TV VINHOS LTDA) em 05/...   14/02/26
13     25  Intimação lido(a) (Para PAYOUT PAGAMENTOS INTE...   03/02/26
14     24            Citação expedido(a) Para TV VINHOS LTDA   2

In [18]:

        
        
abc

NameError: name 'abc' is not defined

In [ ]:
processos = {}
df_relacoes_total = []

for process_json in pasta.glob('*.json'):

    with open(process_json, 'r', encoding='utf-8') as f:
        dados = json.load(f)

    p = process_json.stem

    # =========================
    # MOVIMENTAÇÕES
    # =========================
    df = pd.DataFrame(dados['movimentacoes'])
    processos[p] = df

    df_expedidas = expedidas(df)
    df_lidas = lidas(df)
    df_outros_referenciados = referenciados(df)

    df_expedidas['tipo'] = df_expedidas['ato'].apply(tipo_comunicacao)
    df_lidas['tipo'] = df_lidas['ato'].apply(tipo_comunicacao)

    chave(df_expedidas, df_lidas)
    _relacoes = relacoes(df_expedidas, df_lidas)

    df_relacoes = pd.DataFrame({
        'evento_expedido': _relacoes['evento_expedido'],
        'data_expedicao': _relacoes['data_texto_expedido'],
        'ato_expedido': _relacoes['ato_expedido'],
        'destinatario': _relacoes['destinatario'],
        'meio': _relacoes['meio_real'],
        'canal': _relacoes['meio_comunicacao_expedido'],
        'ato_lido': _relacoes['ato_lido'],
        'data_leitura': _relacoes['data_leitura_str_lido'],
        'evento_lido': _relacoes['evento_lido'],
        'prazo': _relacoes['ato_expedido'].str.extract(r'(\d+\s*dias?)', expand=False)
    })

    df_relacoes = origem(df_relacoes)

    # 🔑 CHAVE ÚNICA GLOBAL
    df_relacoes['key'] = df_relacoes['destinatario'].apply(normalizar_nome)

    # =========================
    # PARTES
    # =========================
    # df_partes = pd.DataFrame(dados['partes'])

    df_partes['key'] = df_partes['nome_normalizado_limpo']#.apply(normalizar_nome)
    df_partes['revelia'] = df_partes['nome'].apply(marcar_revelia)

    df_partes['canais_disponiveis'] = df_partes.apply(canais_disponiveis, axis=1)
    df_partes['canais_disponiveis_lista'] = df_partes.apply(lista_canais_disponiveis, axis=1)

    df_partes['habilitada_receber'] = (
        df_partes['tem_advogado'].fillna(False) |
        df_partes['domicilio_cnj'].fillna(False) |
        df_partes['recebe_intimacao_email'].fillna(False)
    )

    # =========================
    # STATUS PROCESSO
    # =========================
    processo_ok, status = status_processo(df_partes)
    df_relacoes['automatiza'] = status

    # =========================
    # MERGE PARTES ↔ RELAÇÕES
    # =========================
    # df_relacoes = df_relacoes.merge(
    #     df_partes[[
    #         'key',
    #         'revelia',
    #         'email',
    #         'tel',
    #         'habilitada_receber'
    #     ]],
    #     on='key',
    #     how='left'
    # )
    df_relacoes.merge(df_partes, on='key', how='left')

    # =========================
    # CANAIS HISTÓRICOS (CORRETO)
    # =========================
    canais_historicos = (
        df_expedidas.groupby('destinatario')['meio_real']
        .apply(lambda x: ', '.join(sorted(set(filter(pd.notna, x)))))
        .reset_index()
    )

    canais_historicos['key'] = canais_historicos['destinatario'].apply(normalizar_nome)

    df_relacoes = df_relacoes.merge(
        canais_historicos[['key', 'meio_real']],
        on='key',
        how='left'
    )

    df_relacoes = df_relacoes.rename(columns={'meio_real': 'canais_historicos'})

    # =========================
    # CANAL CONFIRMADO (USO REAL)
    # =========================
    canais_confirmados = (
        df_relacoes.groupby('destinatario')['meio']
        .apply(lambda x: ', '.join(sorted(set(filter(pd.notna, x)))))
        .reset_index()
    )

    canais_confirmados['key'] = canais_confirmados['destinatario'].apply(normalizar_nome)

    df_partes = df_partes.merge(
        canais_confirmados[['key', 'meio']],
        on='key',
        how='left'
    )

    df_partes = df_partes.rename(columns={'meio': 'canal_confirmado'})

    # =========================
    # FINAL
    # =========================
    df_relacoes_total.append(df_relacoes)
    # print(
    #     f"Processo: {numero} | "
    #     f"Movimentações: {len(df)}"
    # )
    df_relacoes.drop(['automatiza', 'key'], axis=1, inplace=True)
    display(df_relacoes.set_index('evento_expedido'))
    print(len(df_relacoes))

,data_expedicao,ato_expedido,destinatario,meio,canal,ato_lido,data_leitura,evento_lido,prazo,ato_origem,data_origem,canais_historicos
evento_expedido,,,,,,,,,,,,
60,28/04/26,Intimação expedido(a) Para TV VINHOS LTDA *Ref...,tv vinhos ltda,pessoal,None,Intimação lido(a) (Para TV VINHOS LTDA) em 11/...,2026-05-11,62,NaN,Ato ordinatório praticado,28/04/26,pessoal
58,28/04/26,Intimação expedido(a) (Para PAYOUT PAGAMENTOS ...,payout pagamentos inteligentes ltda,pessoal,None,Intimação lido(a) (Para PAYOUT PAGAMENTOS INTE...,2026-05-08,61,15 dias,Ato ordinatório praticado,28/04/26,"advogado, pessoal"
14,21/01/26,Intimação expedido(a) (Para PAYOUT PAGAMENTOS ...,payout pagamentos inteligentes ltda,pessoal,None,Intimação lido(a) (Para PAYOUT PAGAMENTOS INTE...,2026-02-02,25,NaN,Intimação à disposição,21/01/26,"advogado, pessoal"


3


,data_expedicao,ato_expedido,destinatario,meio,canal,ato_lido,data_leitura,evento_lido,prazo,ato_origem,data_origem,canais_historicos
evento_expedido,,,,,,,,,,,,
219,09/03/26,Intimação expedido(a) (P/ Advgs. de AMIL ASSIS...,amil assistencia medica internacional s a,advogado,domicilio_cnj,Intimação lido(a) (Para AMIL ASSISTENCIA MEDIC...,2026-03-19,223,5 dias,Concedida a Medida Liminar,09/03/26,"advogado, pessoal"
218,09/03/26,Intimação expedido(a) (Para AMIL ASSISTENCIA M...,amil assistencia medica internacional s a,pessoal,None,Intimação lido(a) (Para AMIL ASSISTENCIA MEDIC...,2026-03-19,223,5 dias,Concedida a Medida Liminar,09/03/26,"advogado, pessoal"
174,10/10/25,Intimação expedido(a) (P/ Advgs. de AMIL ASSIS...,amil assistencia medica internacional s a,advogado,domicilio_cnj,Intimação lido(a) (Para AMIL ASSISTENCIA MEDIC...,2025-10-20,182,5 dias,Extinta a execução ou o cumprimento da sentença,10/10/25,"advogado, pessoal"
173,10/10/25,Intimação expedido(a) (Para AMIL ASSISTENCIA M...,amil assistencia medica internacional s a,pessoal,None,Intimação lido(a) (Para AMIL ASSISTENCIA MEDIC...,2025-10-20,182,5 dias,Extinta a execução ou o cumprimento da sentença,10/10/25,"advogado, pessoal"
38,06/05/25,Intimação expedido(a) (P/ Advgs. de AMIL ASSIS...,amil assistencia medica internacional s a,advogado,domicilio_cnj,Intimação lido(a) (Para AMIL ASSISTENCIA MEDIC...,2025-05-16,41,10 dias,Julgada procedente a ação,06/05/25,"advogado, pessoal"
37,06/05/25,Intimação expedido(a) (P/ Advgs. de RAQUEL LEM...,raquel lemos de oliveira,advogado,domicilio_cnj,Intimação lido(a) (Para RAQUEL LEMOS DE OLIVEI...,2025-05-16,40,NaN,Julgada procedente a ação,06/05/25,"advogado, pessoal"
23,17/02/25,Intimação expedido(a) (P/ Advgs. de AMIL ASSIS...,amil assistencia medica internacional s a,advogado,domicilio_cnj,Intimação lido(a) (Para AMIL ASSISTENCIA MEDIC...,2025-03-06,27,10 dias,Intimação à disposição,17/02/25,"advogado, pessoal"
22,17/02/25,Intimação expedido(a) (P/ Advgs. de RAQUEL LEM...,raquel lemos de oliveira,advogado,domicilio_cnj,Intimação lido(a) (Para RAQUEL LEMOS DE OLIVEI...,2025-03-06,26,10 dias,Intimação à disposição,17/02/25,"advogado, pessoal"


8


,data_expedicao,ato_expedido,destinatario,meio,canal,ato_lido,data_leitura,evento_lido,prazo,ato_origem,data_origem,canais_historicos
evento_expedido,,,,,,,,,,,,
117,30/10/24,Intimação expedido(a) (P/ Advgs. de EDILEIDE D...,edileide da silva lima,advogado,domicilio_cnj,Intimação lido(a) (Para EDILEIDE DA SILVA LIMA...,2024-10-31,118,10 dias,Intimação à disposição,30/10/24,advogado
113,16/10/24,Intimação expedido(a) (P/ Advgs. de EDILEIDE D...,edileide da silva lima,advogado,domicilio_cnj,Intimação lido(a) (Para EDILEIDE DA SILVA LIMA...,2024-10-24,114,5 dias,Proferido despacho de mero expediente,16/10/24,advogado
87,05/06/24,Intimação expedido(a) (P/ Advgs. de EDILEIDE D...,edileide da silva lima,advogado,domicilio_cnj,Intimação lido(a) (Para EDILEIDE DA SILVA LIMA...,2024-06-11,93,5 dias,Intimação à disposição,05/06/24,advogado
17,22/03/23,Intimação expedido(a) (P/ Advgs. de EDILEIDE D...,edileide da silva lima,advogado,domicilio_cnj,Intimação lido(a) (Para EDILEIDE DA SILVA LIMA...,2023-03-22,19,10 dias,Julgada procedente em parte a ação,22/03/23,advogado


...

3


,data_expedicao,ato_expedido,destinatario,meio,canal,ato_lido,data_leitura,evento_lido,prazo,ato_origem,data_origem,canais_historicos
evento_expedido,,,,,,,,,,,,
24,02/03/26,Intimação expedido(a) P/ Núcleo de Atuação MP ...,núcleo de atuação mp paulo afonso,None,None,Intimação lido(a) (Para Núcleo de Atuação MP P...,2026-03-06,29,NaN,Intimação para Videoconferência à disposição,02/03/26,
11,12/01/26,Intimação expedido(a) Para CICERO ALENCAR DA S...,cicero alencar da silva,pessoal,None,Intimação lido(a) (Para CICERO ALENCAR DA SILV...,2026-01-21,14,NaN,Intimação para Videoconferência à disposição,12/01/26,pessoal


2


,data_expedicao,ato_expedido,destinatario,meio,canal,ato_lido,data_leitura,evento_lido,prazo,ato_origem,data_origem,canais_historicos
evento_expedido,,,,,,,,,,,,
17,17/12/18,Intimação expedido(a) (P/ Advgs. de OI MOVEL S...,oi movel s a sociedade empresaria em recuperac...,advogado,domicilio_cnj,Intimação lido(a) (Para OI MOVEL S A SOCIEDADE...,2019-01-21,20,NaN,Intimação à disposição,17/12/18,"advogado, pessoal"
16,17/12/18,Intimação expedido(a) (P/ Advgs. de SEVERINO A...,severino alves de oliveira lima,advogado,domicilio_cnj,Intimação lido(a) (Para SEVERINO ALVES DE OLIV...,2019-01-21,19,NaN,Intimação à disposição,17/12/18,advogado


2


In [ ]:
df_final_relacoes = pd.concat(df_relacoes_total, ignore_index=True)
display(df_final_relacoes.head())
df_final_relacoes.columns

,evento_expedido,data_expedicao,ato_expedido,destinatario,meio,canal,ato_lido,data_leitura,evento_lido,prazo,ato_origem,data_origem,canais_historicos
0,60,28/04/26,Intimação expedido(a) Para TV VINHOS LTDA *Ref...,tv vinhos ltda,pessoal,None,Intimação lido(a) (Para TV VINHOS LTDA) em 11/...,2026-05-11,62,NaN,Ato ordinatório praticado,28/04/26,pessoal
1,58,28/04/26,Intimação expedido(a) (Para PAYOUT PAGAMENTOS ...,payout pagamentos inteligentes ltda,pessoal,None,Intimação lido(a) (Para PAYOUT PAGAMENTOS INTE...,2026-05-08,61,15 dias,Ato ordinatório praticado,28/04/26,"advogado, pessoal"
2,14,21/01/26,Intimação expedido(a) (Para PAYOUT PAGAMENTOS ...,payout pagamentos inteligentes ltda,pessoal,None,Intimação lido(a) (Para PAYOUT PAGAMENTOS INTE...,2026-02-02,25,NaN,Intimação à disposição,21/01/26,"advogado, pessoal"
3,219,09/03/26,Intimação expedido(a) (P/ Advgs. de AMIL ASSIS...,amil assistencia medica internacional s a,advogado,domicilio_cnj,Intimação lido(a) (Para AMIL ASSISTENCIA MEDIC...,2026-03-19,223,5 dias,Concedida a Medida Liminar,09/03/26,"advogado, pessoal"
4,218,09/03/26,Intimação expedido(a) (Para AMIL ASSISTENCIA M...,amil assistencia medica internacional s a,pessoal,None,Intimação lido(a) (Para AMIL ASSISTENCIA MEDIC...,2026-03-19,223,5 dias,Concedida a Medida Liminar,09/03/26,"advogado, pessoal"


Index(['evento_expedido', 'data_expedicao', 'ato_expedido', 'destinatario',
       'meio', 'canal', 'ato_lido', 'data_leitura', 'evento_lido', 'prazo',
       'ato_origem', 'data_origem', 'canais_historicos'],
      dtype='object')

In [ ]:
for numero, df in processos.items():
    # df.drop('ato_normalizado')
    
    display(df[df['data_leitura_str'].notnull()])
    # display(df.drop(columns=[['ato_normalizado', 'data_obj']], axis=1))
    print(
        f"Processo: {numero} | "
        f"Movimentações: {len(df)}"
    )


,evento,ato,ato_normalizado,data_texto,data_obj,data_leitura_str,data_referencia_str,autor,observacao,categoria,meio_comunicacao,situacao_comunicacao,destinatario
3,62,Intimação lido(a) (Para TV VINHOS LTDA) em 11/...,intimação lido(a) (para tv vinhos ltda) em 11/...,17/05/26,2026-05-17,2026-05-11,28/04/26,ECT,,intimacao,None,lida,tv vinhos ltda
4,61,Intimação lido(a) (Para PAYOUT PAGAMENTOS INTE...,intimação lido(a) (para payout pagamentos inte...,09/05/26,2026-05-09,2026-05-08,28/04/26,SISTEMA CNJ,,intimacao,None,lida,payout pagamentos inteligentes ltda
10,30,Citação lido(a) P/ PAYOUT PAGAMENTOS INTELIGEN...,citação lido(a) p/ payout pagamentos inteligen...,16/02/26,2026-02-16,2026-02-04,None,ECT,,citacao,None,lida,payout pagamentos inteligentes ltda em 04/02/26
11,28,Intimação lido(a) (Para TV VINHOS LTDA) em 05/...,intimação lido(a) (para tv vinhos ltda) em 05/...,14/02/26,2026-02-14,2026-02-05,21/01/26,ECT,,intimacao,None,lida,tv vinhos ltda
12,26,Citação lido(a) P/ TV VINHOS LTDA em 05/02/26,citação lido(a) p/ tv vinhos ltda em 05/02/26,14/02/26,2026-02-14,2026-02-05,None,ECT,,citacao,None,lida,tv vinhos ltda em 05/02/26
13,25,Intimação lido(a) (Para PAYOUT PAGAMENTOS INTE...,intimação lido(a) (para payout pagamentos inte...,03/02/26,2026-02-03,2026-02-02,21/01/26,SISTEMA CNJ,,intimacao,None,lida,payout pagamentos inteligentes ltda
21,6,Intimação lido(a) (Para DANILO AUGUSTO E ARAUJ...,intimação lido(a) (para danilo augusto e arauj...,15/01/26,2026-01-15,2026-01-15,15/01/26,SISTEMA CNJ,,intimacao,None,lida,danilo augusto e araujo franca


Processo: processo_0000084-87.2026.8.05.0191 | Movimentações: 22


,evento,ato,ato_normalizado,data_texto,data_obj,data_leitura_str,data_referencia_str,autor,observacao,categoria,meio_comunicacao,situacao_comunicacao,destinatario
8,223,Intimação lido(a) (Para AMIL ASSISTENCIA MEDIC...,intimação lido(a) (para amil assistencia medic...,20/03/26,2026-03-20,2026-03-19,09/03/26,SISTEMA CNJ,,intimacao,None,lida,amil assistencia medica internacional s a
16,182,Intimação lido(a) (Para AMIL ASSISTENCIA MEDIC...,intimação lido(a) (para amil assistencia medic...,21/10/25,2025-10-21,2025-10-20,10/10/25,SISTEMA CNJ,,intimacao,None,lida,amil assistencia medica internacional s a
42,41,Intimação lido(a) (Para AMIL ASSISTENCIA MEDIC...,intimação lido(a) (para amil assistencia medic...,17/05/25,2025-05-17,2025-05-16,06/05/25,SISTEMA CNJ,,intimacao,None,lida,amil assistencia medica internacional s a
43,40,Intimação lido(a) (Para RAQUEL LEMOS DE OLIVEI...,intimação lido(a) (para raquel lemos de olivei...,17/05/25,2025-05-17,2025-05-16,06/05/25,SISTEMA CNJ,,intimacao,None,lida,raquel lemos de oliveira
47,27,Intimação lido(a) (Para AMIL ASSISTENCIA MEDIC...,intimação lido(a) (para amil assistencia medic...,07/03/25,2025-03-07,2025-03-06,17/02/25,SISTEMA CNJ,,intimacao,None,lida,amil assistencia medica internacional s a
48,26,Intimação lido(a) (Para RAQUEL LEMOS DE OLIVEI...,intimação lido(a) (para raquel lemos de olivei...,07/03/25,2025-03-07,2025-03-06,17/02/25,SISTEMA CNJ,,intimacao,None,lida,raquel lemos de oliveira
49,25,Intimação lido(a) (Para AMIL ASSISTENCIA MEDIC...,intimação lido(a) (para amil assistencia medic...,18/02/25,2025-02-18,2025-02-17,06/02/25,SISTEMA CNJ,,intimacao,None,lida,amil assistencia medica internacional s a
50,24,Intimação lido(a) (Para RAQUEL LEMOS DE OLIVEI...,intimação lido(a) (para raquel lemos de olivei...,18/02/25,2025-02-18,2025-02-17,06/02/25,SISTEMA CNJ,,intimacao,None,lida,raquel lemos de oliveira
53,13,Citação lido(a) P/ Representante: AMIL ASSISTE...,citação lido(a) p/ representante: amil assiste...,07/02/25,2025-02-07,2025-02-07,None,Tulio Godoy Gomes Salles Rosa,,citacao,None,lida,representante: amil assistencia medica interna...
57,6,Intimação lido(a) (Para RAQUEL LEMOS DE OLIVEI...,intimação lido(a) (para raquel lemos de olivei...,06/02/25,2025-02-06,2025-02-06,06/02/25,SISTEMA CNJ,,intimacao,None,lida,raquel lemos de oliveira


Processo: processo_0000443-71.2025.8.05.0191 | Movimentações: 58


,evento,ato,ato_normalizado,data_texto,data_obj,data_leitura_str,data_referencia_str,autor,observacao,categoria,meio_comunicacao,situacao_comunicacao,destinatario
2,118,Intimação lido(a) (Para EDILEIDE DA SILVA LIMA...,intimação lido(a) (para edileide da silva lima...,31/10/24,2024-10-31,2024-10-31,30/10/24,LEANA BEZERRA GOMES EVANGELISTA,,intimacao,None,lida,edileide da silva lima
4,114,Intimação lido(a) (Para EDILEIDE DA SILVA LIMA...,intimação lido(a) (para edileide da silva lima...,24/10/24,2024-10-24,2024-10-24,16/10/24,LEANA BEZERRA GOMES EVANGELISTA,,intimacao,None,lida,edileide da silva lima
6,93,Intimação lido(a) (Para EDILEIDE DA SILVA LIMA...,intimação lido(a) (para edileide da silva lima...,11/06/24,2024-06-11,2024-06-11,05/06/24,LEANA BEZERRA GOMES EVANGELISTA,,intimacao,None,lida,edileide da silva lima
8,30,Intimação lido(a) (Para LARBELO MOVEIS) em 10/...,intimação lido(a) (para larbelo moveis) em 10/...,16/05/23,2023-05-16,2023-05-10,17/04/23,ECT,,intimacao,None,lida,larbelo moveis
10,19,Intimação lido(a) (Para EDILEIDE DA SILVA LIMA...,intimação lido(a) (para edileide da silva lima...,22/03/23,2023-03-22,2023-03-22,22/03/23,LEANA BEZERRA GOMES EVANGELISTA,,intimacao,None,lida,edileide da silva lima
13,11,Citação lido(a) P/ LARBELO MOVEIS em 08/03/23,citação lido(a) p/ larbelo moveis em 08/03/23,15/03/23,2023-03-15,2023-03-08,None,ECT,,citacao,None,lida,larbelo moveis em 08/03/23
15,5,Intimação lido(a) (Para EDILEIDE DA SILVA LIMA...,intimação lido(a) (para edileide da silva lima...,16/02/23,2023-02-16,2023-02-16,16/02/23,SISTEMA CNJ,,intimacao,None,lida,edileide da silva lima


...

Processo: processo_0004024-31.2024.8.05.0191 | Movimentações: 21


,evento,ato,ato_normalizado,data_texto,data_obj,data_leitura_str,data_referencia_str,autor,observacao,categoria,meio_comunicacao,situacao_comunicacao,destinatario
1,29,Intimação lido(a) (Para Núcleo de Atuação MP P...,intimação lido(a) (para núcleo de atuação mp p...,06/03/26,2026-03-06,2026-03-06,02/03/26,FERNANDO ROGERIO PESSOA VILA NOVA FILHO,,intimacao,None,lida,núcleo de atuação mp paulo afonso
4,14,Intimação lido(a) (Para CICERO ALENCAR DA SILV...,intimação lido(a) (para cicero alencar da silv...,06/02/26,2026-02-06,2026-01-21,12/01/26,ECT,,intimacao,None,lida,cicero alencar da silva
5,13,Intimação lido(a) (Para Núcleo de Atuação MP P...,intimação lido(a) (para núcleo de atuação mp p...,23/01/26,2026-01-23,2026-01-22,None,SISTEMA CNJ,,intimacao,None,lida,núcleo de atuação mp paulo afonso


Processo: processo_0004152-17.2025.8.05.0191 | Movimentações: 8


,evento,ato,ato_normalizado,data_texto,data_obj,data_leitura_str,data_referencia_str,autor,observacao,categoria,meio_comunicacao,situacao_comunicacao,destinatario
0,20,Intimação lido(a) (Para OI MOVEL S A SOCIEDADE...,intimação lido(a) (para oi movel s a sociedade...,21/01/19,2019-01-21,2019-01-21,17/12/18,SISTEMA CNJ,,intimacao,None,lida,oi movel s a sociedade empresaria em recuperac...
1,19,Intimação lido(a) (Para SEVERINO ALVES DE OLIV...,intimação lido(a) (para severino alves de oliv...,21/01/19,2019-01-21,2019-01-21,17/12/18,SISTEMA CNJ,,intimacao,None,lida,severino alves de oliveira lima
5,8,Citação lido(a) P/ OI MOVEL S A SOCIEDADE EMPR...,citação lido(a) p/ oi movel s a sociedade empr...,14/11/18,2018-11-14,2018-11-08,None,ECT,,citacao,None,lida,oi movel s a sociedade empresaria em recuperac...
7,4,Intimação lido(a) (Para SEVERINO ALVES DE OLIV...,intimação lido(a) (para severino alves de oliv...,31/10/18,2018-10-31,2018-10-31,31/10/18,SISTEMA CNJ,,intimacao,None,lida,severino alves de oliveira lima


Processo: processo_0007018-42.2018.8.05.0191 | Movimentações: 8


In [ ]:
display(atos_relacionados[p])

NameError: name 'atos_relacionados' is not defined

In [ ]:
for i, j in atos_relacionados.items():
    display(i, j)

NameError: name 'atos_relacionados' is not defined

In [ ]:
abc

NameError: name 'abc' is not defined

In [ ]:
df_relacoes = pd.DataFrame({

    'ato_origem':
        relacoes['ato_origem'],

    'data_origem':
        relacoes['data_origem'],

    'destinatario':
        relacoes['destinatario_lido'],

    'meio':
        relacoes['meio_real'],

    'canal':
        relacoes['meio_comunicacao_expedido'],

    'evento_lido':
        relacoes['evento_lido'],
        'data_leitura':
        relacoes['data_leitura_str_lido'],
     'prazo':
        relacoes['ato_expedido'].str.extract(
            r'(\d+\s*dias?)',
            expand=False
        ),

    'evento_expedido':
        relacoes['evento_expedido'],

    'data_expedicao':
        relacoes['data_texto_expedido'],

   

    'ato_expedido':
        relacoes['ato_expedido'],

    'ato_lido':
        relacoes['ato_lido']
})

TypeError: 'function' object is not subscriptable

In [ ]:
display(df_relacoes)

,evento_expedido,data_expedicao,ato_expedido,destinatario,meio,canal,ato_lido,data_leitura,evento_lido,prazo,ato_origem,data_origem,key,automatiza,canais_historicos
0,17,17/12/18,Intimação expedido(a) (P/ Advgs. de OI MOVEL S...,oi movel s a sociedade empresaria em recuperac...,advogado,domicilio_cnj,Intimação lido(a) (Para OI MOVEL S A SOCIEDADE...,2019-01-21,20,NaN,Intimação à disposição,17/12/18,oi movel s a sociedade empresaria em recuperac...,automatizar,"advogado, pessoal"
1,16,17/12/18,Intimação expedido(a) (P/ Advgs. de SEVERINO A...,severino alves de oliveira lima,advogado,domicilio_cnj,Intimação lido(a) (Para SEVERINO ALVES DE OLIV...,2019-01-21,19,NaN,Intimação à disposição,17/12/18,severino alves de oliveira lima,automatizar,advogado


In [ ]:
for destinatario, grupo in df.groupby('destinatario'):

    grupo = grupo.sort_values('data_obj')
    display(grupo)

,evento,ato,ato_normalizado,data_texto,data_obj,data_leitura_str,data_referencia_str,autor,observacao,categoria,meio_comunicacao,situacao_comunicacao,destinatario
6,6,Citação expedido(a) Para OI MOVEL S A SOCIEDAD...,citação expedido(a) para oi movel s a sociedad...,31/10/18,2018-10-31,None,None,MARCELO PEREIRA DA SILVAA,,citacao,None,expedida,oi movel s a sociedade empresaria em recuperac...
2,17,Intimação expedido(a) (P/ Advgs. de OI MOVEL S...,intimação expedido(a) (p/ advgs. de oi movel s...,17/12/18,2018-12-17,None,None,IVAN GUEDES DA SILVA,,intimacao,domicilio_cnj,expedida,oi movel s a sociedade empresaria em recuperac...
0,20,Intimação lido(a) (Para OI MOVEL S A SOCIEDADE...,intimação lido(a) (para oi movel s a sociedade...,21/01/19,2019-01-21,2019-01-21,17/12/18,SISTEMA CNJ,,intimacao,None,lida,oi movel s a sociedade empresaria em recuperac...


,evento,ato,ato_normalizado,data_texto,data_obj,data_leitura_str,data_referencia_str,autor,observacao,categoria,meio_comunicacao,situacao_comunicacao,destinatario
5,8,Citação lido(a) P/ OI MOVEL S A SOCIEDADE EMPR...,citação lido(a) p/ oi movel s a sociedade empr...,14/11/18,2018-11-14,2018-11-08,None,ECT,,citacao,None,lida,oi movel s a sociedade empresaria em recuperac...


,evento,ato,ato_normalizado,data_texto,data_obj,data_leitura_str,data_referencia_str,autor,observacao,categoria,meio_comunicacao,situacao_comunicacao,destinatario
7,4,Intimação lido(a) (Para SEVERINO ALVES DE OLIV...,intimação lido(a) (para severino alves de oliv...,31/10/18,2018-10-31,2018-10-31,31/10/18,SISTEMA CNJ,,intimacao,None,lida,severino alves de oliveira lima
3,16,Intimação expedido(a) (P/ Advgs. de SEVERINO A...,intimação expedido(a) (p/ advgs. de severino a...,17/12/18,2018-12-17,None,None,IVAN GUEDES DA SILVA,,intimacao,domicilio_cnj,expedida,severino alves de oliveira lima
1,19,Intimação lido(a) (Para SEVERINO ALVES DE OLIV...,intimação lido(a) (para severino alves de oliv...,21/01/19,2019-01-21,2019-01-21,17/12/18,SISTEMA CNJ,,intimacao,None,lida,severino alves de oliveira lima


In [ ]:
lidas = grupo[
    grupo['situacao_comunicacao'] == 'lida'
]
expedidas = grupo[
    grupo['situacao_comunicacao'] == 'expedida'
]

In [ ]:
display(lidas)

,evento,ato,ato_normalizado,data_texto,data_obj,data_leitura_str,data_referencia_str,autor,observacao,categoria,meio_comunicacao,situacao_comunicacao,destinatario
7,4,Intimação lido(a) (Para SEVERINO ALVES DE OLIV...,intimação lido(a) (para severino alves de oliv...,31/10/18,2018-10-31,2018-10-31,31/10/18,SISTEMA CNJ,,intimacao,None,lida,severino alves de oliveira lima
1,19,Intimação lido(a) (Para SEVERINO ALVES DE OLIV...,intimação lido(a) (para severino alves de oliv...,21/01/19,2019-01-21,2019-01-21,17/12/18,SISTEMA CNJ,,intimacao,None,lida,severino alves de oliveira lima


In [ ]:
resultado = []

for _, lido in lidas.iterrows():

    expedido = expedidas[
        (
            expedidas['destinatario']
            == lido['destinatario']
        )
        &
        (
            expedidas['data_obj']
            ==
            pd.to_datetime(
                lido['data_referencia_str'],
                dayfirst=True
            )
        )
    ]

    if len(expedido):

        expedido = expedido.iloc[0]

        resultado.append({

            'destinatario':
                lido['destinatario'],

            'evento_expedido':
                expedido['evento'],

            'evento_lido':
                lido['evento'],

            'data_expedicao':
                expedido['data_obj'],

            'data_leitura':
                lido['data_obj'],

            'meio':
                expedido['meio_comunicacao']
        })

In [ ]:
print(resultado)

[]


In [ ]:
def buscar_relacao_evento(ato):
    
    padrao = re.search(
        # r'Intimação\s+(lido\(a\)|expedido\(a\))'
        # r'\s+(lido\(a\)|expedido\(a\))'
        r'(.+?)\s+(lido\(a\)|expedido\(a\))' #qualquer documento lido ou expedido
        r'.*?\(Para\s+(.*?)\)'
        r'.*?em\s+(\d{2}/\d{2}/\d{2})',
        # r'\(P/\s+Advgs?\.\s+de\s+(.*?)\)',
        # r'.*?\*Referente ao evento\s+(.*?)\((\d{2}/\d{2}/\d{2})\)',
        ato,
        re.I | re.S
    )
    return padrao
    
    



In [ ]:
processos = []
for process_json in pasta.glob('*.json'):
    with open(process_json, 'r', encoding='utf-8') as f:
        processo = json.load(f)
        processos.append(processo)
# print(len(processos))
        df = pd.DataFrame(processos['movimentacoes'])
    display(df)



TypeError: list indices must be integers or slices, not str

In [ ]:
def referencia(ato):
    m_referente = re.search(r'\*Referente ao evento\s+(.*?)\((\d{2}/\d{2}/\d{2})\)', ato,re.I)

    if m_referente:
        evento = m_referente.group(1).strip()
        data_referencia = m_referente.group(2)

        print(f'''ref: {evento}
data referência {data_referencia}''')
        print('')
    return m_referente
        # print(data_referencia)
        
def destinatario(ato):

    #advogado
    m_adv = re.search(r'\(P/\s+Advgs?\.\s+de\s+(.*?)\)', ato, re.I)
    if m_adv:
        return {'tipo': 'advogado', 'nome': m_adv.group(1).strip()}
    # Advogado

    # DJEN
    m_djen = re.search(r'\bDJEN\b',ato,re.I)
    if m_djen:
        return {'tipo': 'djen', 'nome': m_djen.group(0).strip()}

    # AR
    m_ar = re.search(r'Juntada\s+de\s+AR',ato,re.I)
    if m_ar:
        return {'tipo': 'ar', 'nome': m_ar.group(0).strip()}
   

    # parte
    m = re.search(r'\(Para\s+(.*?)\)',ato,re.I)

    if m: 
        return {'tipo': 'parte','nome': m.group(1).strip()}

    return None

In [ ]:
eventos_encadeados = []

for processo in processos:

    for mov in processo['movimentacoes']:

        # 1. Precisa ser um evento lido
        rel_lido = buscar_relacao_evento(mov['ato'])

        if not rel_lido:
            continue

        if rel_lido.group(2) != 'lido(a)':
            continue

        destinatario_lido = rel_lido.group(3)

        # 2. Precisa ter referência ao ato origem
        ref = referencia(mov['ato'])

        if not ref:
            continue

        nome_origem = ref.group(1).strip()
        data_origem = ref.group(2)

        # 3. Encontrar o ato origem
        ato_origem = None

        for mov_ref in processo['movimentacoes']:

            if mov_ref['data_texto'] != data_origem:
                continue

            if nome_origem.lower() in mov_ref['ato'].lower():
                ato_origem = mov_ref
                break

        if not ato_origem:
            continue

ref: Ato ordinatório praticado
data referência 28/04/26

ref: Ato ordinatório praticado
data referência 28/04/26

ref: Intimação à disposição
data referência 21/01/26

ref: Intimação à disposição
data referência 21/01/26

ref: Audiência de Conciliação Designada (Telepresencial)
data referência 15/01/26

ref: Concedida a Medida Liminar
data referência 09/03/26

ref: Extinta a execução ou o cumprimento da sentença
data referência 10/10/25

ref: Julgada procedente a ação
data referência 06/05/25

ref: Julgada procedente a ação
data referência 06/05/25

ref: Intimação à disposição
data referência 17/02/25

ref: Intimação à disposição
data referência 17/02/25

ref: Não Concedida a Medida Liminar a RAQUEL LEMOS DE OLIVEIRA
data referência 06/02/25

ref: Não Concedida a Medida Liminar a RAQUEL LEMOS DE OLIVEIRA
data referência 06/02/25

ref: Audiência  de Conciliação Designada (Telepresencial)
data referência 06/02/25

ref: Intimação à disposição
data referência 30/10/24

ref: Proferido despa

In [ ]:
eventos_encadeados = []

for processo in processos:

    for mov_lido in processo['movimentacoes']:

        # 1 - localizar leitura
        rel_lido = buscar_relacao_evento(mov_lido['ato'])

        if not rel_lido:
            continue

        if rel_lido.group(2) != 'lido(a)':
            continue

        destinatario_lido = rel_lido.group(3)

        # 2 - localizar referência
        ref = referencia(mov_lido['ato'])

        if not ref:
            continue

        nome_origem = ref.group(1).strip()
        data_origem = ref.group(2)

        # 3 - localizar ato origem
        ato_origem = next(
            (
                m for m in processo['movimentacoes']
                if m['data_texto'] == data_origem
                and nome_origem.lower() in m['ato'].lower()
            ),
            None
        )

        if not ato_origem:
            continue

        # 4 - localizar expedição correspondente
        evento_expedido = None
        meio_expedido = None

        for mov_exp in processo['movimentacoes']:

            rel_exp = buscar_relacao_evento(mov_exp['ato'])

            if not rel_exp:
                continue

            if rel_exp.group(2) != 'expedido(a)':
                continue

            info_exp = destinatario(mov_exp['ato'])

            if not info_exp:
                continue

            # mesmo destinatário
            if info_exp['nome'] != destinatario_lido:
                continue

            evento_expedido = mov_exp
            meio_expedido = info_exp['tipo']
            break

        # 5 - montar cadeia
        eventos_encadeados.append({
            'ato_origem': ato_origem,
            'evento_expedido': evento_expedido,
            'evento_lido': mov_lido,

            'destinatario': destinatario_lido,

            'meio_expedido': meio_expedido,
            'meio_lido': 'parte',

            'evento_referenciado': nome_origem,
            'data_referencia': data_origem,
        })


ref: Ato ordinatório praticado
data referência 28/04/26

ref: Ato ordinatório praticado
data referência 28/04/26

ref: Intimação à disposição
data referência 21/01/26

ref: Intimação à disposição
data referência 21/01/26

ref: Audiência de Conciliação Designada (Telepresencial)
data referência 15/01/26

ref: Concedida a Medida Liminar
data referência 09/03/26

ref: Extinta a execução ou o cumprimento da sentença
data referência 10/10/25

ref: Julgada procedente a ação
data referência 06/05/25

ref: Julgada procedente a ação
data referência 06/05/25

ref: Intimação à disposição
data referência 17/02/25

ref: Intimação à disposição
data referência 17/02/25

ref: Não Concedida a Medida Liminar a RAQUEL LEMOS DE OLIVEIRA
data referência 06/02/25

ref: Não Concedida a Medida Liminar a RAQUEL LEMOS DE OLIVEIRA
data referência 06/02/25

ref: Audiência  de Conciliação Designada (Telepresencial)
data referência 06/02/25

ref: Intimação à disposição
data referência 30/10/24

ref: Proferido despa

In [ ]:
eventos_encadeados = []

for processo in processos:
    for mov in processo['movimentacoes']:
        if 'lido(a)' in mov['ato']:
            print(f"Ev FILHO :{mov['evento']} - {mov['ato']}")
            evento_relacionado_filho = buscar_relacao_evento(mov['ato'])

            if evento_relacionado_filho:
                data_referencia = referencia(mov['ato']).group(2)
                destinatario_filho = buscar_relacao_evento(mov['ato']).group(3)
                # if not destinatario_filho:
                #     continue
            if data_referencia:
                for mov_ref in processo['movimentacoes']:
                    if mov_ref['data_texto'] == data_referencia:
                        evento_relacionado_pai = buscar_relacao_evento(mov_ref['ato'])
                        mov_pai = mov_ref
                        # dados_evento_pai = buscar_relacao_evento(mov_pai)
                        # if dados_evento_pai:
                        #     print(dados_evento_pai.group(2))
                if mov_pai:
                    evento_relacionado_pai = buscar_relacao_evento(mov_pai['ato'])
                    if evento_relacionado_pai:
                        documento = evento_relacionado_pai.group(1)
                        situacao = evento_relacionado_pai.group(2)
                        destinatario_pai = evento_relacionado_pai.group(3)
                        if not destinatario_pai and destinatario_filho:
                            continue
                        # if documento:# == evento_relacionado_filho.group(1):
                        if destinatario_pai == destinatario_filho:

                            print(f"Ev PAI: Ev:{mov_pai['evento']} - {mov_pai['data_texto']} - {mov_pai['ato']}") 
                            print(f"documento: {documento}")
                            print(f"situacão: {situacao}")
                            print(f"destinatário: {destinatario_pai}")
                            print('')

            if evento_relacionado_filho:
                if evento_relacionado_filho.group(3):
                    print("destinatário:", evento_relacionado_filho.group(3))
                if evento_relacionado_filho.group(1):
                    print("documento:", evento_relacionado_filho.group(1))
                if evento_relacionado_filho.group(2):
                    print("situação:", evento_relacionado_filho.group(2))
                if evento_relacionado_filho.group(4):
                    print("data leitura:", evento_relacionado_filho.group(4))


                else:
                    continue
                
            print('---------------------------------')
            print('')
            # if referencias:
            #     referencia = referencias.group(1)
            #     data_referencia = referencia.group(2)
                
                
                # print(referencia)
                # print(data_referencia)
                # print('---------------------------------')
                # print('')
        #     evento_relacionado = buscar_relacao_evento(mov['ato'])
        #     if not evento_relacionado:
        #         continue
        #     if evento_relacionado.group(4):
        #         data_ato_ref = evento_relacionado.group(4)
        #         for mov_ref in processo["movimentacoes"]:
        #             if mov_ref["data_texto"] == data_ato_ref:
        #                 eventos_pai = mov_ref
        #                 eventos_encadeados.append(eventos_pai)
        #                 eventos_encadeados.append(mov['ato'])
        #                 print(eventos_encadeados)
        #                 print('---------------------------------------------------')
        #                 break
        #     # else: 
        #     #     continue
                
        #     # if evento_relacionado:
        #     #     print("situação:", evento_relacionado.group(1))
        #     #     print("destinatário:", evento_relacionado.group(2))
        #     #     print("data leitura:", evento_relacionado.group(3))
        #     #     print("ato referência:", evento_relacionado.group(4))
        #     #     print("data ato:", evento_relacionado.group(5))
        #     #     print('---------------------------------------------------')

        # #     destinatario_por_adv = re.search(r'P\/\s+Advgs?\.', mov['ato'])
        # #     destinatario_pessoalmente = re.search(r'([A-Z])', mov['ato'])
        # # if destinatario_por_adv:
        # #     print(destinatario_por_adv.group())

        # # if destinatario_pessoalmente:
        # #     print(destinatario_pessoalmente.group())

        # print(mov['evento'])
        # print(mov['ato'])

    # if 'lido' in mov['ato']:
        # destinatario = 
        
        # print(f'DATA:{mov["data_texto"]}')
        # print(f'CAT:{mov["categoria"]}')
    # print(f'SIT:{mov["situacao_comunicacao"]})
    
    # print(f'REF:{mov["data_referencia_str"]})

Ev FILHO :62 - Intimação lido(a) (Para TV VINHOS LTDA) em 11/05/26 *Referente ao evento  Ato ordinatório praticado(28/04/26)
ref: Ato ordinatório praticado
data referência 28/04/26

destinatário: TV VINHOS LTDA
documento: Intimação
situação: lido(a)
data leitura: 11/05/26
---------------------------------

Ev FILHO :61 - Intimação lido(a) (Para PAYOUT PAGAMENTOS INTELIGENTES LTDA) em 08/05/26 *Referente ao evento  Ato ordinatório praticado(28/04/26)
ref: Ato ordinatório praticado
data referência 28/04/26

destinatário: PAYOUT PAGAMENTOS INTELIGENTES LTDA
documento: Intimação
situação: lido(a)
data leitura: 08/05/26
---------------------------------

Ev FILHO :30 - Citação lido(a) P/ PAYOUT PAGAMENTOS INTELIGENTES LTDA em 04/02/26
---------------------------------

Ev FILHO :28 - Intimação lido(a) (Para TV VINHOS LTDA) em 05/02/26 *Referente ao evento Intimação à disposição(21/01/26)
ref: Intimação à disposição
data referência 21/01/26

destinatário: TV VINHOS LTDA
documento: Intimação


In [ ]:
eventos_encadeados = []

for processo in processos:
    print( 'Processo: ', processo)
    for mov in processo['movimentacoes']:
        # if 'lido(a)' in mov['ato']:
        #     print(f"Ev FILHO :{mov['evento']} - {mov['ato']}")
        evento_relacionado_filho = buscar_relacao_evento(mov['ato'])

        if evento_relacionado_filho:
            data_referencia = referencia(mov['ato']).group(2)
            destinatario_filho = buscar_relacao_evento(mov['ato']).group(3)
            # meio_filho = destinatario(mov['ato'])
            
            # if meio_filho:
            #     print(meio_filho['tipo'])

            if not destinatario_filho:
                continue
        if data_referencia:
            for mov_ref in processo['movimentacoes']:
                if mov_ref['data_texto'] == data_referencia:
                    evento_relacionado_pai = buscar_relacao_evento(mov_ref['ato'])
                    meio_pai = destinatario(mov_ref['ato'])
                    mov_pai = mov_ref
                        # dados_evento_pai = buscar_relacao_evento(mov_pai)
                        # if dados_evento_pai:
                        #     print(dados_evento_pai.group(2))
            if mov_pai:
                evento_relacionado_pai = buscar_relacao_evento(mov_ref['ato'])
                if evento_relacionado_pai:
                    documento = evento_relacionado_pai.group(1)
                    situacao = evento_relacionado_pai.group(2)
                    destinatario_pai = evento_relacionado_pai.group(3)
                    
                    if not destinatario_pai and destinatario_filho:
                        continue
                        # if documento:# == evento_relacionado_filho.group(1):
                    if destinatario_pai == destinatario_filho:

                        print(f"Ev PAI: Ev:{mov_pai['evento']} - {mov_pai['data_texto']} - {mov_pai['ato']}") 
                        print(f"documento: {documento}")
                        print(f"situacão: {situacao}")
                        print(f"destinatário: {destinatario_pai}")
                        if meio_pai:
                            print(f"meio: {meio_pai['tipo']}")
                        print('')

        if evento_relacionado_filho:
            print(f"Ev FILHO: Ev:{mov['evento']} - {mov['data_texto']} - {mov['ato']}") 
            if evento_relacionado_filho.group(3):
                print("destinatário:", evento_relacionado_filho.group(3))
            if evento_relacionado_filho.group(1):
                print("documento:", evento_relacionado_filho.group(1))
            if evento_relacionado_filho.group(2):
                print("situação:", evento_relacionado_filho.group(2))
            if evento_relacionado_filho.group(4):
                print("data leitura:", evento_relacionado_filho.group(4))


            else:
                continue
                
        print('---------------------------------')
        print('')


Processo:  {'partes': [{'nome': 'DANILO AUGUSTO E ARAUJO FRANCA', 'nome_normalizado': 'danilo augusto e araujo franca', 'cpf/cnpj': '891.424.205-63', 'tipo': 'EXEQUENTE', 'papel': 'PROMOVENTE', 'recebe_intimacao_email': False, 'domicilio_cnj': False, 'tem_advogado': True, 'email': 'FRANCA_DANILO@HOTMAIL.COM', 'tel': '71996123344', 'logradouro': 'ENDEREÇO RUA NELSON RODRIGUES DO NASCIMENTO', 'numero': '45', 'complemento': 'PANORAMA', 'bairro': '(CONTATO:', 'cidade': 'PAULO AFONSO', 'uf': 'BA', 'cep': '48605041'}, {'nome': 'PAYOUT PAGAMENTOS INTELIGENTES LTDA(Rev. Arg)', 'nome_normalizado': 'payout pagamentos inteligentes ltda(rev. arg)', 'cpf/cnpj': '57.225.538/0001-11', 'tipo': 'EXECUTADO', 'papel': 'PROMOVIDO', 'recebe_intimacao_email': False, 'domicilio_cnj': True, 'tem_advogado': True, 'email': None, 'tel': None, 'logradouro': 'ENDEREÇO ALAMEDA RIO NEGRO', 'numero': '503', 'complemento': 'ALPHAVILLE CENTRO INDUSTRIAL E EMPRESARIAL ALPHAVILLE', 'bairro': None, 'cidade': 'BARUERI', 'u

In [ ]:
eventos_encadeados = []

for processo in processos:

    for mov in processo["movimentacoes"]:

        # if "lido(a)" not in mov["ato"]:
        #     continue

        evento_relacionado = buscar_relacao_evento(mov["ato"])

        if not evento_relacionado:
            continue

        destinatario = evento_relacionado.group(2)
        data_leitura = evento_relacionado.group(3)
        ato_referencia = evento_relacionado.group(4)
        # data_ato_ref = evento_relacionado.group(5)

        for mov_ref in processo["movimentacoes"]:

            # if (
            #     mov_ref["data_texto"] == data_ato_ref
            #     and ato_referencia.lower() in mov_ref["ato"].lower()
            # ):

            eventos_encadeados.append({
                    "evento_filho": mov["evento"],
                    "ato_filho": mov["ato"],

                    "evento_pai": mov_ref["evento"],
                    "ato_pai": mov_ref["ato"],

                    "destinatario": destinatario,
                    "data_leitura": data_leitura,
                    # "data_referencia": data_ato_ref,
            })
                
            break
# print(eventos_encadeados['evento_pai'])
# print(eventos_encadeados['ato_filho'])


In [ ]:
# for rel in eventos_encadeados:
#     print(
#         f'Evento {rel["evento_filho"]} '
#         f'-> Evento {rel["evento_pai"]}'
#     )

In [ ]:
decisao_sentenca = (
    'Inexistência de bens penhoráveis','Procedência em Parte', 'Procedência', 'Improcedência',
    'Extinção da execução ou do cumprimento da sentença', 'Ausência do autor à audiência', 'Homologada a Transação',
    'Inadmissibilidade do procedimento sumaríssimo', 'Liminar', 'Acolhimento e Embargos de Declaração',
)
for e in eventos_encadeados:
    if e in decisao_sentenca:
        print(f"[FILHO] {e['evento_filho']} | {e['ato_filho']}")
        print(f"[PAI  ] {e['evento_pai']} | {e['ato_pai']}")
        print(f"Destinatário: {e['destinatario']}  | Leitura: {e['data_leitura']}  | Ref: {e['data_referencia']}")
        print("-" * 80)

In [ ]:
padroes_sentenca = (
    'procedente',
    'procedência',
    'improcedente',
    'liminar',
    'extinta a execução',
    'extinto o processo',
    'homologada a transação',
)


for e in eventos_encadeados:
    evento = e['evento_filho'].lower()

    if any(p.lower() in evento for p in padroes_sentenca):
        print(f"[FILHO] {e['evento_filho']} | {e['ato_filho']}")
        print(f"[PAI  ] {e['evento_pai']} | {e['ato_pai']}")
        print(f"Destinatário: {e['destinatario']}")
        print("-" * 80)

In [ ]:
eventos_encadeados = []

for processo in processos:

    for mov in processo["movimentacoes"]:

        if "lido(a)" not in mov["ato"]:
            continue

        evento_relacionado = buscar_relacao_evento(mov["ato"])

        if not evento_relacionado:
            continue

        destinatario = evento_relacionado.group(2)
        data_leitura = evento_relacionado.group(3)
        ato_referencia = evento_relacionado.group(4)
        data_ato_ref = evento_relacionado.group(5)

        for mov_ref in processo["movimentacoes"]:

            if (
                mov_ref["data_texto"] == data_ato_ref
                and ato_referencia.lower() in mov_ref["ato"].lower()
            ):

                eventos_encadeados.append({
                    "evento_filho": mov["evento"],
                    "ato_filho": mov["ato"],

                    "evento_pai": mov_ref["evento"],
                    "ato_pai": mov_ref["ato"],

                    "destinatario": destinatario,
                    "data_leitura": data_leitura,
                    "data_referencia": data_ato_ref,
                })
                
                break
# print(eventos_encadeados['evento_pai'])
# print(eventos_encadeados['ato_filho'])


IndexError: no such group

In [ ]:
for processo in processos:

    for mov_lido in processo['movimentacoes']:

        # 1. Precisa ser uma leitura
        rel_lido = buscar_relacao_evento(mov_lido['ato'])

        if not rel_lido:
            continue

        if rel_lido.group(2) != 'lido(a)':
            continue

        # 2. Precisa ter referência ao evento principal
        ref = referencia(mov_lido['ato'])

        if not ref:
            continue

        nome_evento = ref.group(1)
        data_referencia = ref.group(2)

        destinatario = rel_lido.group(3)

        # 3. Encontrar o evento principal
        evento_principal = next(
            (
                m for m in processo['movimentacoes']
                if m['data_texto'] == data_referencia
            ),
            None
        )

        if not evento_principal:
            continue

        # 4. Encontrar a expedição correspondente
        evento_expedido = None

        for mov_exp in processo['movimentacoes']:

            rel_exp = buscar_relacao_evento(mov_exp['ato'])

            if not rel_exp:
                continue

            if rel_exp.group(2) != 'expedido(a)':
                continue

            if rel_exp.group(3) == destinatario:
                evento_expedido = mov_exp
                break

        # 5. Montar o encadeamento
        eventos_encadeados.append({
            'evento_principal': evento_principal,
            'evento_expedido': evento_expedido,
            'evento_lido': mov_lido,

            'nome_evento': nome_evento,
            'destinatario': destinatario,
            'data_referencia': data_referencia,
            'data_leitura': rel_lido.group(4),
        })

ref: Ato ordinatório praticado
data referência 28/04/26

ref: Ato ordinatório praticado
data referência 28/04/26

ref: Intimação à disposição
data referência 21/01/26

ref: Intimação à disposição
data referência 21/01/26

ref: Audiência de Conciliação Designada (Telepresencial)
data referência 15/01/26

ref: Concedida a Medida Liminar
data referência 09/03/26

ref: Extinta a execução ou o cumprimento da sentença
data referência 10/10/25

ref: Julgada procedente a ação
data referência 06/05/25

ref: Julgada procedente a ação
data referência 06/05/25

ref: Intimação à disposição
data referência 17/02/25

ref: Intimação à disposição
data referência 17/02/25

ref: Não Concedida a Medida Liminar a RAQUEL LEMOS DE OLIVEIRA
data referência 06/02/25

ref: Não Concedida a Medida Liminar a RAQUEL LEMOS DE OLIVEIRA
data referência 06/02/25

ref: Audiência  de Conciliação Designada (Telepresencial)
data referência 06/02/25

ref: Intimação à disposição
data referência 30/10/24

ref: Proferido despa

In [ ]:
for e in eventos_encadeados:

    print(
        f"PRINCIPAL: {e['evento_principal']['evento']} - "
        f"{e['evento_principal']['ato']}"
    )

    if e['evento_expedido']:
        print(
            f"EXPEDIDO : {e['evento_expedido']['evento']} - "
            f"{e['evento_expedido']['ato']}"
        )

    print(
        f"LIDO     : {e['evento_lido']['evento']} - "
        f"{e['evento_lido']['ato']}"
    )

    print('-' * 80)

PRINCIPAL: 60 - Intimação expedido(a) Para TV VINHOS LTDA *Referente ao evento  Ato ordinatório praticado(28/04/26)
LIDO     : 62 - Intimação lido(a) (Para TV VINHOS LTDA) em 11/05/26 *Referente ao evento  Ato ordinatório praticado(28/04/26)
--------------------------------------------------------------------------------
PRINCIPAL: 60 - Intimação expedido(a) Para TV VINHOS LTDA *Referente ao evento  Ato ordinatório praticado(28/04/26)
LIDO     : 61 - Intimação lido(a) (Para PAYOUT PAGAMENTOS INTELIGENTES LTDA) em 08/05/26 *Referente ao evento  Ato ordinatório praticado(28/04/26)
--------------------------------------------------------------------------------
PRINCIPAL: 14 - Intimação expedido(a) (Para PAYOUT PAGAMENTOS INTELIGENTES LTDA) Sem prazo
LIDO     : 28 - Intimação lido(a) (Para TV VINHOS LTDA) em 05/02/26 *Referente ao evento Intimação à disposição(21/01/26)
--------------------------------------------------------------------------------
PRINCIPAL: 14 - Intimação expedido(a) (

In [ ]:
rel_lido = buscar_relacao_evento(mov['ato'])

if not rel_lido:
    continue

if rel_lido.group(2) != 'lido(a)':
    continue

In [ ]:
ato_origem = None

for mov_ref in processo['movimentacoes']:

    if mov_ref['data_texto'] != data_origem:
        continue

    if evento_origem_nome.lower() not in mov_ref['ato'].lower():
        continue

    ato_origem = mov_ref
    break

In [ ]:
processos = {}
atos_relacionados = {}
df_relacoes = pd.DataFrame()
for process_json in pasta.glob('*.json'):
    with open(process_json, 'r', encoding='utf-8') as f:
        dados = json.load(f)
        # padrao_revelia = r'\(rev\.\s*arg\.?\)'
        # partes = dados['partes']
        # nome = dados['partes']['nome']
        # nome = re.sub(padrao_revelia,'',nome,flags=re.I).strip()
        # nome_normalizado = dados['partes']['nome_normalizado']
        # nome_normalizado = re.sub(padrao_revelia,'',nome_normalizado, flags=re.I).strip()
        # revelia = bool(re.search(padrao_revelia, nome, re.I))
        
        df_partes = pd.DataFrame(partes)
        df_partes['nome_normalizado_2'] = df_partes['nome'].apply(normalizar_nome)

        df_partes['canais_disponiveis'] = (df_partes.apply(canais_disponiveis, axis=1))
        df_partes['canais_disponiveis_lista'] = (df_partes.apply(lista_canais_disponiveis, axis=1))
        df_partes['habilitada_receber'] = (df_partes['tem_advogado'].fillna(False) | df_partes['domicilio_cnj'].fillna(False)
                                           | df_partes['recebe_intimacao_email'].fillna(False))
        processo_ok = df_partes['habilitada_receber'].all()
        autores_ok = (df_partes.loc[df_partes['papel'] == 'PROMOVENTE','habilitada_receber'].all())
        autores_pendentes = df_partes[(df_partes['papel'] == 'PROMOVENTE')& (~df_partes['habilitada_receber'])]
        acusados_ok = (df_partes.loc[df_partes['papel'] == 'PROMOVIDO','habilitada_receber'].all())
        acusados_pendentes = df_partes[(df_partes['papel'] == 'PROMOVIDO')& (~df_partes['habilitada_receber'])]
        
        df_partes['all_autors'] = df_partes['papel'] == 'PROMOVENTE' 
        df_partes['all_accused'] = df_partes['papel'] == 'PROMOVIDO'
        # status_processo = True if processo_ok else False
        
        
        
        # pprint(partes_resumo)
        p = process_json.stem
        processos[p] = pd.DataFrame(dados['movimentacoes'])
        df = processos[p]
            
       
        df_expedidas = expedidas(processos[p])
        df_expedidas = expedidas(df)
        print("ANTES")
        print(df_partes.columns.tolist())

        canais_historicos = (df_expedidas.groupby('destinatario')['meio_real'].unique().reset_index())
        canais_historicos['canais_historicos'] = (canais_historicos['meio_real'].apply(
        lambda x: ', '.join(sorted(set(i for i in x if pd.notna(i))))))

        # df_partes['canais_historicos'] = (canais_historicos['meio_real'].apply(lambda x: ', '.join(
        #             sorted(set(i for i in x if pd.notna(i))))))
        
        print(canais_historicos.columns.tolist())

        df_partes = df_partes.merge(canais_historicos[['destinatario', 'canais_historicos']],
                                    left_on='nome_normalizado',right_on='destinatario',how='left')
        print("DEPOIS")
        print(df_partes.columns.tolist())
        # print(df_partes)
        # canais_historicos['canais_historicos'] = (canais_historicos['meio_real'].apply(lambda x: ', '.join(
        #             sorted(set(i for i in x if pd.notna(i))))))
        # print(canais_historicos.head())
        

        df_expedidas['tipo'] = df_expedidas['ato'].apply(tipo_comunicacao)
        df_lidas = lidas(processos[p])
        df_lidas['tipo'] = df_lidas['ato'].apply(tipo_comunicacao)
        
        chave(df_expedidas, df_lidas)
        _relacoes = relacoes(df_expedidas, df_lidas)
        # print(_relacoes.columns.tolist())
        df_relacoes = pd.DataFrame({
            
            'evento_expedido': _relacoes['evento_expedido'],
            'data_expedicao': _relacoes['data_texto_expedido'],
            'ato_expedido': _relacoes['ato_expedido'],
            'destinatario': _relacoes['destinatario'],

            'meio': _relacoes['meio_real'],
            'canal': _relacoes['meio_comunicacao_expedido'],

            'ato_lido': _relacoes['ato_lido'],
            'data_leitura': _relacoes['data_leitura_str_lido'],
            'evento_lido': _relacoes['evento_lido'],

            'prazo': _relacoes['ato_expedido'].str.extract(
                r'(\d+\s*dias?)',
                expand=False
            )
        })
        

        df_relacoes = origem(df_relacoes)
        df_relacoes['destinatario_norm'] = df_relacoes['destinatario'].apply(normalizar_nome)
        df_relacoes['automatiza'] = status_processo(autores_ok, acusados_ok)
        # df_relacoes = df_relacoes.merge(canais_historicos[['destinatario', 'canais_historicos']],
        #                                 on='destinatario',how='left')
        df_relacoes = df_relacoes.merge(
            df_partes[
                [
                    'nome_normalizado',
                    'revelia',
                    # 'domicilio_cnj',
                    # 'recebe_intimacao_email',
                    'email',
                    'canais_historicos',
                    'tel',
                ]
            ],
            left_on='destinatario_norm',
            right_on='nome_normalizado_2',
            how='left'
        )#.drop(columns=['nome_normalizado'], errors='ignore')
        # df_relacoes['canais_disponiveis'] = (df_relacoes.apply(canais_disponiveis, axis=1))
        # df_relacoes['possui_email'] = (df_relacoes['email'].notna())
        canais_confirmados = (df_relacoes.groupby('destinatario')['meio'].unique().reset_index())
        print(df_relacoes.columns.tolist())
        print(df_partes.columns.tolist())
        print(canais_historicos.columns.tolist())
        canais_confirmados['canal_confirmado'] = (canais_confirmados['meio'].apply(lambda x: ', '.join(
            sorted(set(i for i in x if pd.notna(i))))))
        df_partes = df_partes.merge(canais_confirmados[['destinatario', 'canal_confirmado']],
                                    left_on='nome_normalizado',right_on='destinatario',how='left')
        # df_partes = df_partes.merge(canais_confirmados[['destinatario', 'canais_confirmado']],
        #                             left_on='nome_normalizado',right_on='destinatario',how='left')

        # df_relacoes['status_comunicacao'] = df_relacoes.apply(status_comunicacao, axis=1)
        # df_relacoes['automatiza'] = status_processo(autores_ok, acusados_ok)
        # df_relacoes['autores_ok'] = autores_ok
        # df_relacoes['acusados_ok'] = acusados_ok
        # print(df_relacoes[df_relacoes['tem_advogado'].isna()][['destinatario']].drop_duplicates())
        # print(
        #     df_partes[
        #         df_partes['nome_normalizado']
        #         .str.contains('tv vinhos', case=False, na=False)
        #     ]
        # )

        atos_relacionados[p] = df_relacoes
        display(atos_relacionados[p])